# 파인튜닝 실험 조종석 (Experiment Registry)

**흐름**: 모든 실험은 같은 사이클이다 — `학습(train.py) → 평가(eval --adapter) → experiments.csv 비교`.
- **유형 1 (데이터/하이퍼파라미터)**: 레지스트리에서 aug_mult, lr, epochs 등만 다르게
- **유형 2 (프롬프트)**: `scripts/prompts.py`에 후보 추가 → 레지스트리에 `prompt="이름"` — 학습·평가에 **자동으로 세트 적용**됨

**사용법**: "모두 실행"만 하면 스모크 → 본학습(백그라운드) → 진행 게이지 → 평가 → 분석까지 자동 완주. 실험 추가는 ①레지스트리에 한 줄.

**주의**
- GPU는 하나 = **한 번에 한 실험**. 학습 셀 실행 중 다른 GPU 커널 금지
- 본학습은 백그라운드 프로세스라 커널이 죽어도 학습은 계속됨 (단, 뒤 셀 자동 진행은 끊김 — 그 경우 아침에 ⑤부터 다시 실행)
- 밤샘: 전원 연결 + 덮개 열기 + 창 유지
- 판정 기준은 항상 **acc_shuffled**. 무작위 4.2%, ±4%p 이내는 노이즈 (상세: `EXPERIMENTS.md`)

In [1]:
# ① 실험 레지스트리 — 실험 추가 = 여기 한 줄. 기본값과 다른 것만 적는다
DEFAULTS = dict(model="./models/Qwen3-VL-2B-Instruct", load_4bit=False,
                aug_mult=2, lr=1e-4, epochs=1, max_hours=10, prompt="v1_list", extra="")

EXPERIMENTS = {
    "exp01_aug2_lr1e4": {},                            # 기준점 (전부 기본값)
    "exp02_aug4":       dict(aug_mult=4),               # 증강 배수
    "exp03_lr5e5":      dict(lr=5e-5),                  # 학습률
    "exp04_r32":        dict(extra="--lora-r 32 --lora-alpha 64"),  # LoRA rank
    "exp05_v2temporal": dict(prompt="v2_temporal"),     # 프롬프트 실험 (학습·평가 세트 자동)
    # "exp10_4b":       dict(model="./models/Qwen3-VL-4B-Instruct", load_4bit=True),  # 레시피 확정 후
}

def cfg(name): return {**DEFAULTS, **EXPERIMENTS[name]}

def train_cmd(name, smoke=False):
    c = cfg(name)
    run = name + ("_smoke" if smoke else "")
    cmd = (f"python scripts/train.py --run-name {run} --model {c['model']} "
           f"--aug-mult {c['aug_mult']} --lr {c['lr']} --epochs {c['epochs']} "
           f"--max-hours {c['max_hours']} --prompt {c['prompt']} {c['extra']}").strip()
    if c["load_4bit"]: cmd += " --load-4bit"
    if smoke: cmd += " --max-samples 12 --grad-accum 4 --max-steps 5"
    return cmd

def eval_cmd(name):
    c = cfg(name)
    cmd = (f"python scripts/eval_zero_shot.py --model {c['model']} "
           f"--adapter ./outputs/runs/{name}/adapter --prompt {c['prompt']}")
    if c["load_4bit"]: cmd += " --load-4bit"
    return cmd

for n in EXPERIMENTS: print(f"[{n}]\n  학습: {train_cmd(n)}\n  평가: {eval_cmd(n)}")

[exp01_aug2_lr1e4]
  학습: python scripts/train.py --run-name exp01_aug2_lr1e4 --model ./models/Qwen3-VL-2B-Instruct --aug-mult 2 --lr 0.0001 --epochs 1 --max-hours 10 --prompt v1_list
  평가: python scripts/eval_zero_shot.py --model ./models/Qwen3-VL-2B-Instruct --adapter ./outputs/runs/exp01_aug2_lr1e4/adapter --prompt v1_list
[exp02_aug4]
  학습: python scripts/train.py --run-name exp02_aug4 --model ./models/Qwen3-VL-2B-Instruct --aug-mult 4 --lr 0.0001 --epochs 1 --max-hours 10 --prompt v1_list
  평가: python scripts/eval_zero_shot.py --model ./models/Qwen3-VL-2B-Instruct --adapter ./outputs/runs/exp02_aug4/adapter --prompt v1_list
[exp03_lr5e5]
  학습: python scripts/train.py --run-name exp03_lr5e5 --model ./models/Qwen3-VL-2B-Instruct --aug-mult 2 --lr 5e-05 --epochs 1 --max-hours 10 --prompt v1_list
  평가: python scripts/eval_zero_shot.py --model ./models/Qwen3-VL-2B-Instruct --adapter ./outputs/runs/exp03_lr5e5/adapter --prompt v1_list
[exp04_r32]
  학습: python scripts/train.py --run-name 

In [2]:
# ② 실험 선택
NAME = "exp01_aug2_lr1e4"

In [3]:
# ③ 스모크 (~3분) — 코드/설정 바꿨으면 본학습 전에 반드시
!{train_cmd(NAME, smoke=True)}

학습 제외: ./splits/holdout_300.csv (300개 누적)
학습 제외: ./eda/stratified_valid.csv (494개 누적)
기반 12개 x 증강 2 = 학습 항목 24개
trainable params: 6,422,528 || all params: 2,133,954,560 || trainable%: 0.3010
[최종] 어댑터 저장 -> ./outputs/runs\exp01_aug2_lr1e4_smoke\adapter

종료(max_steps(5) 도달): 5 스텝, 20 항목, 0.01시간, peak VRAM 6.41GB
다음: python scripts/eval_zero_shot.py --model ./models/Qwen3-VL-2B-Instruct --adapter ./outputs/runs\exp01_aug2_lr1e4_smoke\adapter



Loading weights: 100%|██████████| 625/625 [00:01<00:00, 514.50it/s]
W0713 14:44:50.328000 23124 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels

epoch 1/1:   0%|          | 0/24 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.

epoch 1/1:  79%|███████▉  | 19/24 [00:19<00:05,  1.04s/it, loss=0.375, vram=6.35]


In [4]:
# ④ 본학습 시작 — 백그라운드 프로세스로 실행 (커널을 막지 않음 → 아래 게이지 셀이 동작)
import subprocess, os
os.makedirs(f"./outputs/runs/{NAME}", exist_ok=True)
console = open(f"./outputs/runs/{NAME}/console.log", "w", encoding="utf-8")
proc = subprocess.Popen(train_cmd(NAME), shell=True, stdout=console, stderr=subprocess.STDOUT,
                        env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(f"학습 시작: PID {proc.pid} | 콘솔 로그: outputs/runs/{NAME}/console.log")
print("중단하려면: 새 셀에서 proc.kill() (또는 작업관리자에서 위 PID 트리 종료)")

학습 시작: PID 22188 | 콘솔 로그: outputs/runs/exp01_aug2_lr1e4/console.log
중단하려면: 새 셀에서 proc.kill() (또는 작업관리자에서 위 PID 트리 종료)


In [ ]:
# ⑤ 진행 게이지 — 30초마다 갱신, 학습이 끝나면 자동으로 다음 셀(평가)로 진행
#    ("모두 실행" 시 이 셀이 학습 완료까지 대기하는 역할을 겸한다)
import sys, time
from IPython.display import clear_output
sys.path.insert(0, "scripts")
from watch_train import read_status, render

run_dir = f"./outputs/runs/{NAME}"
p = globals().get("proc")  # 커널 재시작 후에도 게이지만 다시 돌릴 수 있게
while True:
    try:
        c, last = read_status(run_dir)
        line = render(c, last)
        done = (p is not None and p.poll() is not None) or \
               (last is not None and last.opt_step >= c["total_opt_steps"])
    except FileNotFoundError:
        line, done = "시작 대기 중... (모델 로드에 1~2분)", False
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), line)
    if done:
        tail = open(f"{run_dir}/console.log", encoding="utf-8", errors="replace").read().splitlines()
        print("\n학습 종료 — 콘솔 로그 마지막:\n" + "\n".join(tail[-3:]))
        break
    time.sleep(30)

23:01:47 [############------------------]  41.6%  7520/18080 항목 | loss 0.129 | 3.93초/항목 | 경과 493분, 남은 11.5시간 | 완료 예상 10:33 (내일 기준) | VRAM 6.99GB


In [ ]:
# ⑥ holdout 300 평가 (~5분) — experiments.csv에 자동 누적
!{eval_cmd(NAME)}

In [ ]:
# ⑦ loss 곡선 — 전체 run 오버레이 (학습 도중에도 실행 가능)
import pandas as pd, matplotlib.pyplot as plt, glob, os
fig, ax = plt.subplots(figsize=(9, 4))
for f in sorted(glob.glob("./outputs/runs/*/train_log.csv")):
    run = os.path.basename(os.path.dirname(f))
    if run.endswith("_smoke") or run == "smoke": continue
    log = pd.read_csv(f)
    ax.plot(log.opt_step, log.loss, label=f"{run} (vram {log.peak_vram_gb.max()}GB)")
ax.set_xlabel("opt step"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("학습 곡선 비교")
plt.tight_layout(); plt.show()

In [ ]:
# ⑧ 실험 비교표 — acc_shuffled(핵심 지표) 기준 정렬
import pandas as pd
exp = pd.read_csv("./outputs/experiments.csv")
cols = ["timestamp", "model_id", "prompt", "accuracy", "acc_shuffled", "acc_identity", "parse_fail", "sec_per_sample", "peak_vram_gb"]
display(exp[[c for c in cols if c in exp.columns]].sort_values("acc_shuffled", na_position="first"))
print("기준선: 무작위 섞인샘플 4.2% | ±4%p 이내는 노이즈 | zero-shot Qwen3-VL-2B: 전체 14.0%/섞인 0.8%")

In [ ]:
# ⑨ 오답 비교 — run_b 기준으로 run_a 대비 새로 맞춘/틀린 샘플 (run_a=None이면 zero-shot과 비교)
import pandas as pd, os

def errors_of(run=None, model=DEFAULTS["model"]):
    base = os.path.basename(model)
    path = f"./outputs/errors_{base}.csv" if run is None else f"./outputs/errors_{base}_{run}.csv"
    return pd.read_csv(path)

def compare(run_b, run_a=None):
    a, b = errors_of(run_a), errors_of(run_b)
    fixed, broken = set(a.Id) - set(b.Id), set(b.Id) - set(a.Id)
    print(f"오답 {len(a)}({run_a or 'zero-shot'}) -> {len(b)}({run_b}) | 새로 맞춤 {len(fixed)} / 새로 틀림 {len(broken)}")
    print("\n[새로 맞춘 샘플]"); display(a[a.Id.isin(fixed)][["Id", "Sentence", "gt"]].head(10))
    print("[여전히/새로 틀리는 샘플 — 텍스트 EDA 공유용]"); display(b.head(10))

compare(NAME)  # NAME vs zero-shot / 실험끼리는 compare("exp02_aug4", "exp01_aug2_lr1e4")

## 로드맵 (상세: `EXPERIMENTS.md` §5)

1. **exp01 기준점 먼저** — 이거 없이는 아무것도 비교 불가
2. 하이퍼파라미터: 증강(exp02) → lr(exp03) → rank(exp04) — 변수 하나씩, 밤 배치 1회 = 실험 1개
3. 프롬프트: `prompts.py`에 후보 추가 → 미니 학습(`--max-samples 1000 --max-steps 300`, ~1.5h)으로 스크리닝 → 승자만 밤 배치
4. 출력 형식(24-순열 분류 등)은 프롬프트만으로 안 됨 — train.py의 target_text + eval 파서 동시 수정 필요
5. 레시피 확정 → 4B 스케일업 (lr만 재탐색)